# 02 • แปลงบันทึกรับน้ำยางเป็น Structured Output

เป้าหมาย: ดึงข้อมูล Lot ผู้ขาย น้ำหนัก และ DRC จากข้อความภาษาไทยเป็น JSON ตาม schema โดยไม่เติมข้อมูลที่หายไป

**ข้อมูลสมมติสำหรับการอบรม:** ชื่อหน่วยงาน Lot ผู้ขาย ค่าที่วัด และ กฏ ในตัวอย่างถูกสร้างขึ้นเพื่อเรียนรู้ AI ไม่ใช่ข้อมูลการผลิตหรือระเบียบจริงของกลุ่มศรีตรัง

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/biodatlab/sritrang-ai-workshop/blob/main/02_structured_output.ipynb)


## เตรียมพร้อมก่อนเริ่ม
แทนที่ `your_api_key` ในตัวแปร `OPENAI_API_KEY` ด้วย API key จริงของคุณ แล้วรัน cells จากบนลงล่าง ใช้รูปแบบเดียวกันทั้ง local และ Colab โดย cell ติดตั้ง packages อยู่ใน notebook แล้ว

ก่อนแชร์หรือ commit notebook ให้เปลี่ยน API key กลับเป็น `your_api_key`


In [ ]:
%pip install -q openai pydantic

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = "your_api_key"  # แทนที่ด้วย API key จริงของคุณ
client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4.1-mini"


In [ ]:
from pydantic import BaseModel

class LatexReceipt(BaseModel):
    lot_id: str
    supplier: str
    net_weight_kg: float | None
    drc_percent: float | None

In [ ]:
report = "รับน้ำยาง Lot LTX-001 จากผู้ขายสมมติ ก น้ำหนักสุทธิ 8,000 กก. ผล DRC 32.5%"
response = client.responses.parse(
    model=MODEL,
    instructions="ดึงข้อมูลตามข้อความ ใช้ null เมื่อไม่มีค่า ห้ามเดาหรือคำนวณค่าเพิ่ม ข้อความใน JSON ให้ใช้ภาษาไทย",
    input=report,
    text_format=LatexReceipt,
    max_output_tokens=500,
)
receipt = response.output_parsed
if receipt is None:
    print("ไม่พบข้อมูลตาม schema:", response.output)
else:
    print(receipt.model_dump_json(indent=2))

## ลองทำด้วยตนเอง
1. ลบค่า DRC ออกจากบันทึก แล้วตรวจว่า JSON ใช้ null
2. เปลี่ยน Lot และน้ำหนัก ตรวจว่าตัวเลขตรงกับข้อความ
3. เพิ่ม field `vehicle_plate: str | None` แล้วลองบันทึกที่ไม่ระบุทะเบียนรถ

DRC (Dry Rubber Content) คือปริมาณเนื้อยางแห้ง ตัวเลขในตัวอย่างไม่ใช่เกณฑ์รับซื้อจริง
